# 02 — Clean and Standardize the Seattle Tree Asset Inventory

Creates a working a working tree feaure class, checks key attributes, removes exact asset IDs, standardizes size and age variables, and creates a clearly labeled **simulated inspection record** where public condition/ histroy values are unavailable. 

> **Environment:** Run these notebooks inside an ArcGIS Pro cloned Python environemnt with `arcpy`, `pandas`, `numpy`, `matplotlib`, `requests`, and ArcGIS API for Python installed.

The workflow is a **portfolio simulation of municipal asset management**. It prioritizes trees for inspection; it does **not** predct failure or replace an ISA-certified arborist's assessment.

This notebook:

- Preserves the original DBH value.
- Imputes missing DBH values transparently.
- Derives a DBH-based maturity class as a life-stage/age proxy.
- Creates explicitly labeled simulated condition and inspection-history fields.
- Produces QA/QC and imputation summaries.

> DBH-based maturity is not verified chronological age. Growth rates vary by species, site, maintenance, and environmental conditions.

In [ ]:
from pathlib import Path
import os
from datetime import datetime
import hashlib
import numpy as np
import pandas as pd
import arcpy

arcpy.env.overwriteOutput = True

PROJECT_ROOT = Path(os.environ.get("SEATTLE_TREE_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
GDB = PROJECT_ROOT / "data" / "Seattle_Tree_Assets.gdb"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_TREES = str(GDB / "spr_trees")
TREES_CLEAN = str(GDB / "trees_clean")

if not arcpy.Exists(SOURCE_TREES):
    raise FileNotFoundError(SOURCE_TREES)

if arcpy.Exists(TREES_CLEAN):
    arcpy.management.Delete(TREES_CLEAN)

arcpy.management.CopyFeatures(SOURCE_TREES, TREES_CLEAN)

TREE_ID_FIELD = "TREE_ID"
SPECIES_FIELD = "SPECIES"
COMMON_FIELD = "COMMON"
DBH_FIELD = "DBH_WL"
PARK_FIELD = "PARK"

available = {f.name for f in arcpy.ListFields(TREES_CLEAN)}

required = [TREE_ID_FIELD, SPECIES_FIELD, COMMON_FIELD, DBH_FIELD]
missing_required = [f for f in required if f not in available]
if missing_required:
    raise RuntimeError(f"Missing required fields: {missing_required}")

print(f"Copied {int(arcpy.management.GetCount(TREES_CLEAN)[0]):,} tree records.")

In [ ]:
# Remove exact duplicate source IDs while retaining records with null IDs.
before_count = int(arcpy.management.GetCount(TREES_CLEAN)[0])
arcpy.management.DeleteIdentical(TREES_CLEAN, [TREE_ID_FIELD])
after_count = int(arcpy.management.GetCount(TREES_CLEAN)[0])

print(f"Duplicates removed: {before_count - after_count:,}")
print(f"Remaining records:   {after_count:,}")

In [ ]:
def add_field_if_missing(fc, name, field_type, length=None, alias=None):
    existing = {f.name for f in arcpy.ListFields(fc)}
    if name not in existing:
        kwargs = {}
        if length:
            kwargs["field_length"] = length
        if alias:
            kwargs["field_alias"] = alias
        arcpy.management.AddField(fc, name, field_type, **kwargs)

new_fields = [
    ("ASSET_ID", "TEXT", 40, "Asset ID"),
    ("ASSET_TYPE", "TEXT", 30, "Asset Type"),
    ("DBH_ORIG", "DOUBLE", None, "Original DBH"),
    ("DBH_CLEAN", "DOUBLE", None, "Analysis DBH"),
    ("DBH_IMPUTED", "SHORT", None, "DBH Was Imputed"),
    ("DBH_METHOD", "TEXT", 50, "DBH Value Method"),
    ("DBH_SCORE", "DOUBLE", None, "DBH Priority Score"),
    ("MATURITY", "TEXT", 25, "Derived Maturity Class"),
    ("MATURITY_SC", "DOUBLE", None, "Maturity Priority Score"),
    ("AGE_PROXY", "TEXT", 100, "Age Proxy Description"),
    ("SIM_COND", "TEXT", 20, "Simulated Condition"),
    ("COND_SCORE", "DOUBLE", None, "Condition Priority Score"),
    ("LAST_INSP", "DATE", None, "Simulated Last Inspection"),
    ("INSP_SOURCE", "TEXT", 40, "Inspection Data Source"),
    ("DATA_OK", "SHORT", None, "Core Data Passed QA"),
    ("QA_NOTES", "TEXT", 500, "QA Notes"),
]

for args in new_fields:
    add_field_if_missing(TREES_CLEAN, *args)

print("Standardized fields added.")

In [ ]:
# Build reference medians from valid observed DBH values.
source_fields = ["OID@", TREE_ID_FIELD, SPECIES_FIELD, COMMON_FIELD, DBH_FIELD]
if PARK_FIELD in available:
    source_fields.append(PARK_FIELD)

records = []
with arcpy.da.SearchCursor(TREES_CLEAN, source_fields) as cursor:
    for row in cursor:
        records.append(dict(zip(source_fields, row)))

dbh_df = pd.DataFrame(records)
dbh_df["DBH_VALID"] = pd.to_numeric(dbh_df[DBH_FIELD], errors="coerce")
dbh_df.loc[dbh_df["DBH_VALID"] <= 0, "DBH_VALID"] = np.nan

observed_df = dbh_df.dropna(subset=["DBH_VALID"]).copy()
citywide_median = float(observed_df["DBH_VALID"].median())

def grouped_median_lookup(dataframe, group_field, minimum_count):
    if group_field not in dataframe.columns:
        return {}
    grouped = (
        dataframe.dropna(subset=[group_field])
        .groupby(group_field)["DBH_VALID"]
        .agg(["median", "count"])
    )
    grouped = grouped[grouped["count"] >= minimum_count]
    return grouped["median"].to_dict()

species_medians = grouped_median_lookup(observed_df, SPECIES_FIELD, 5)
common_medians = grouped_median_lookup(observed_df, COMMON_FIELD, 5)
park_medians = (
    grouped_median_lookup(observed_df, PARK_FIELD, 10)
    if PARK_FIELD in observed_df.columns else {}
)

print(f"Observed DBH:        {len(observed_df):,}")
print(f"Missing/invalid DBH: {dbh_df['DBH_VALID'].isna().sum():,}")
print(f"Citywide median DBH: {citywide_median:.2f}")

In [ ]:
def classify_maturity(dbh):
    if dbh is None or pd.isna(dbh):
        return "Unknown", 0.0
    dbh = float(dbh)
    if dbh < 6:
        return "Young", 20.0
    if dbh < 18:
        return "Established", 50.0
    if dbh < 30:
        return "Mature", 80.0
    return "Large Mature", 100.0

def scale_dbh_score(dbh):
    if dbh is None or pd.isna(dbh):
        return 0.0
    return round(min(max(float(dbh), 0.0) / 40.0 * 100.0, 100.0), 2)

def stable_rng(asset_value):
    digest = hashlib.sha256(str(asset_value).encode("utf-8")).hexdigest()
    return np.random.default_rng(int(digest[:8], 16))

In [ ]:
REFERENCE_DATE = datetime(2026, 1, 1)

cursor_fields = [
    "OID@", TREE_ID_FIELD, SPECIES_FIELD, COMMON_FIELD, DBH_FIELD
]
if PARK_FIELD in available:
    cursor_fields.append(PARK_FIELD)

cursor_fields += [
    "ASSET_ID", "ASSET_TYPE", "DBH_ORIG", "DBH_CLEAN",
    "DBH_IMPUTED", "DBH_METHOD", "DBH_SCORE",
    "MATURITY", "MATURITY_SC", "AGE_PROXY",
    "SIM_COND", "COND_SCORE", "LAST_INSP", "INSP_SOURCE",
    "DATA_OK", "QA_NOTES"
]

with arcpy.da.UpdateCursor(TREES_CLEAN, cursor_fields) as rows:
    for row in rows:
        values = dict(zip(cursor_fields, row))
        oid = values["OID@"]
        tree_id = values.get(TREE_ID_FIELD)
        species = values.get(SPECIES_FIELD)
        common = values.get(COMMON_FIELD)
        park = values.get(PARK_FIELD) if PARK_FIELD in cursor_fields else None

        original_dbh = pd.to_numeric(values.get(DBH_FIELD), errors="coerce")
        if pd.isna(original_dbh) or original_dbh <= 0:
            original_dbh = None
        else:
            original_dbh = float(original_dbh)

        asset_id = (
            f"SPR-{tree_id}"
            if tree_id not in (None, "")
            else f"SPR-OID-{oid}"
        )

        notes = []

        if original_dbh is not None:
            clean_dbh = original_dbh
            imputed = 0
            method = "Observed source value"
        elif species not in (None, "") and species in species_medians:
            clean_dbh = float(species_medians[species])
            imputed = 1
            method = "Species median"
            notes.append("DBH imputed from scientific-species median")
        elif common not in (None, "") and common in common_medians:
            clean_dbh = float(common_medians[common])
            imputed = 1
            method = "Common-name median"
            notes.append("DBH imputed from common-name median")
        elif park not in (None, "") and park in park_medians:
            clean_dbh = float(park_medians[park])
            imputed = 1
            method = "Park median"
            notes.append("DBH imputed from park median")
        else:
            clean_dbh = citywide_median
            imputed = 1
            method = "Citywide median"
            notes.append("DBH imputed from citywide median")

        dbh_score = scale_dbh_score(clean_dbh)
        maturity, maturity_score = classify_maturity(clean_dbh)
        age_proxy = (
            f"{maturity} life-stage proxy derived from DBH; "
            f"not verified chronological age"
        )

        rng = stable_rng(asset_id)
        size_stress = 0.55 * dbh_score + 0.45 * maturity_score
        p_poor = min(0.05 + size_stress / 600.0, 0.28)
        p_fair = min(0.25 + size_stress / 700.0, 0.48)
        draw = rng.random()

        if draw < p_poor:
            condition, condition_score = "Poor", 100.0
        elif draw < p_poor + p_fair:
            condition, condition_score = "Fair", 60.0
        else:
            condition, condition_score = "Good", 20.0

        years_since = int(rng.integers(0, 9))
        last_insp = datetime(REFERENCE_DATE.year - years_since, 1, 1)

        if tree_id in (None, ""):
            notes.append("Generated asset ID because TREE_ID is missing")
        if species in (None, ""):
            notes.append("Scientific species missing")
        if common in (None, ""):
            notes.append("Common name missing")
        if not notes:
            notes.append("Passed core checks with observed DBH")

        updates = {
            "ASSET_ID": asset_id,
            "ASSET_TYPE": "Municipal Tree",
            "DBH_ORIG": original_dbh,
            "DBH_CLEAN": clean_dbh,
            "DBH_IMPUTED": imputed,
            "DBH_METHOD": method,
            "DBH_SCORE": dbh_score,
            "MATURITY": maturity,
            "MATURITY_SC": maturity_score,
            "AGE_PROXY": age_proxy,
            "SIM_COND": condition,
            "COND_SCORE": condition_score,
            "LAST_INSP": last_insp,
            "INSP_SOURCE": "SIMULATED portfolio field",
            "DATA_OK": 1,
            "QA_NOTES": "; ".join(notes),
        }

        for field_name, value in updates.items():
            row[cursor_fields.index(field_name)] = value

        rows.updateRow(row)

print(f"Prepared {int(arcpy.management.GetCount(TREES_CLEAN)[0]):,} tree assets.")

In [ ]:
print("Source and analysis DBH completeness:")

summary = pd.DataFrame({
    "field": [
        DBH_FIELD,
        "DBH_ORIG",
        "DBH_CLEAN",
        "MATURITY",
    ],
    "purpose": [
        "Original Seattle source field",
        "Preserved observed source value",
        "Observed or imputed analysis value",
        "Derived from DBH_CLEAN",
    ],
    "missing_count": [
        validation_df[DBH_FIELD].isna().sum(),
        validation_df["DBH_ORIG"].isna().sum(),
        validation_df["DBH_CLEAN"].isna().sum(),
        validation_df["MATURITY"].isna().sum(),
    ],
})

summary["missing_pct"] = (
    summary["missing_count"]
    / len(validation_df)
    * 100
).round(2)

display(summary)
assert validation_df["DBH_CLEAN"].isna().sum() == 0
assert validation_df["MATURITY"].isna().sum() == 0

print(
    "Analysis fields are complete. "
    "Missing values remain only in preserved source fields."
)

In [ ]:
source_dbh_numeric = pd.to_numeric(
    validation_df[DBH_FIELD],
    errors="coerce",
)

null_source_count = source_dbh_numeric.isna().sum()

nonpositive_source_count = (
    source_dbh_numeric.notna()
    & (source_dbh_numeric <= 0)
).sum()

imputed_count = (
    validation_df["DBH_IMPUTED"] == 1
).sum()

print(f"Null source DBH values:        {null_source_count:,}")
print(f"Non-positive source DBH:       {nonpositive_source_count:,}")
print(f"Total records requiring fill:  {imputed_count:,}")

In [ ]:
# ---------------------------------------------------------------
# QA COMPLETENESS REPORT
# ---------------------------------------------------------------

qa_fields = [
    TREE_ID_FIELD,
    SPECIES_FIELD,
    COMMON_FIELD,
    DBH_FIELD,
    "DBH_CLEAN",
    "DBH_IMPUTED",
    "MATURITY",
    "SIM_COND",
]

qa_records = []
total = int(arcpy.management.GetCount(TREES_CLEAN)[0])

for field_name in qa_fields:
    values = [row[0]
        for row in arcpy.da.SearchCursor(TREES_CLEAN, [field_name],)]
    missing = sum(value is None or (isinstance(value, str) and not value.strip()) for value in values)
    qa_records.append({
        "field": field_name,
        "total_records": total,
        "missing_count": missing,
        "missing_pct": (
            round(missing / total * 100, 2,) 
            if total
            else None),})

qa_df = pd.DataFrame(qa_records)

action_lookup = {
    TREE_ID_FIELD: ("Generated a temporary asset ID where the source ID was missing"),
    SPECIES_FIELD: ("Retained as null; no species value was inferred"),
    COMMON_FIELD: ("Retained as null; no common-name value was inferred"),
    DBH_FIELD: ("Preserved as the source value; missing or invalid values " "were imputed into DBH_CLEAN"),
    "DBH_CLEAN": ("Completed using observed DBH or the documented " "imputation hierarchy"),
    "DBH_IMPUTED": ("Flagged records whose analysis DBH was estimated"),
    "MATURITY": ("Derived from DBH_CLEAN as a life-stage proxy"),
    "SIM_COND": ("Populated as an explicitly labeled simulated portfolio field"),
}

qa_df["action_taken"] = (qa_df["field"].map(action_lookup).fillna("Reviewed; no additional treatment applied"))
qa_df = qa_df[["field", "total_records",  "missing_count", "missing_pct", "action_taken",]]
qa_df.to_csv(OUTPUT_DIR / "tree_inventory_qa_revised.csv",index=False,)

# ---------------------------------------------------------------
# DBH IMPUTATION SUMMARY
# ---------------------------------------------------------------

imputation_summary = (validation_df.groupby(["DBH_IMPUTED","DBH_METHOD",],dropna=False,).size().reset_index(name="tree_count"))
imputation_summary["share_pct"] = (imputation_summary["tree_count"]/ total* 100).round(2)
imputation_summary.to_csv(OUTPUT_DIR / "dbh_imputation_summary.csv",index=False,)

# ---------------------------------------------------------------
# MATURITY SUMMARY
# ---------------------------------------------------------------

maturity_summary = (validation_df.groupby("MATURITY",dropna=False,).size().reset_index(name="tree_count"))
maturity_summary["share_pct"] = (maturity_summary["tree_count"] / total * 100).round(2)
maturity_summary.to_csv(OUTPUT_DIR / "maturity_class_summary.csv",index=False,)

# ---------------------------------------------------------------
# DATA TREATMENT SUMMARY
# ---------------------------------------------------------------

treatment_counts = {
    "Original DBH retained": 0,
    "Species median imputation": 0,
    "Common-name median imputation": 0,
    "Park median imputation": 0,
    "Citywide median imputation": 0,
    "Generated asset IDs": 0,
}


with arcpy.da.SearchCursor(TREES_CLEAN,[TREE_ID_FIELD,"DBH_IMPUTED","DBH_METHOD",],) as cursor:
    for (tree_id, dbh_imputed,dbh_method,) in cursor:
        if dbh_imputed == 0:
            treatment_counts["Original DBH retained"] += 1

        elif dbh_method == "Species median":
            treatment_counts["Species median imputation"] += 1

        elif dbh_method == "Common-name median":
            treatment_counts["Common-name median imputation"] += 1

        elif dbh_method == "Park median":
            treatment_counts["Park median imputation"] += 1

        elif dbh_method == "Citywide median":
            treatment_counts["Citywide median imputation"] += 1

        if tree_id in (None, ""):
            treatment_counts["Generated asset IDs"] += 1


treatment_summary = pd.DataFrame([
        {"treatment": treatment,"record_count": count,}
        for treatment, count in treatment_counts.items()])


treatment_summary["share_pct"] = (treatment_summary["record_count"] / total * 100).round(2)
treatment_summary.to_csv(OUTPUT_DIR / "data_treatment_summary.csv", index=False,)

# ---------------------------------------------------------------
# DISPLAY REPORTS
# ---------------------------------------------------------------

print("QA completeness and treatment report:")
display(qa_df)

print("\nDBH imputation summary:")
display(imputation_summary)

print("\nMaturity summary:")
display(maturity_summary)

print("\nData treatment summary:")
display(treatment_summary)

In [ ]:
# ---------------------------------------------------------------
# Data Treatment Summary
# ---------------------------------------------------------------

treatment_counts = {
    "Original DBH retained": 0,
    "Species median imputation": 0,
    "Common-name median imputation": 0,
    "Park median imputation": 0,
    "Citywide median imputation": 0,
    "Generated asset IDs": 0,
}

treatment_fields = [TREE_ID_FIELD, "DBH_IMPUTED", "DBH_METHOD",]

with arcpy.da.SearchCursor(TREES_CLEAN, treatment_fields,) as cursor:
    for tree_id, dbh_imputed, dbh_method in cursor:
        if dbh_imputed == 0:
            treatment_counts["Original DBH retained"] += 1

        elif dbh_method == "Species median":
            treatment_counts["Species median imputation"] += 1

        elif dbh_method == "Common-name median":
            treatment_counts["Common-name median imputation"] += 1

        elif dbh_method == "Park median":
            treatment_counts["Park median imputation"] += 1

        elif dbh_method == "Citywide median":
            treatment_counts["Citywide median imputation"] += 1

        if tree_id in (None, ""):
            treatment_counts["Generated asset IDs"] += 1


treatment_summary = pd.DataFrame([{"treatment": treatment,"record_count": count,}
        for treatment, count
        in treatment_counts.items()])

treatment_summary["share_pct"] = (treatment_summary["record_count"] / total * 100).round(2)
treatment_summary.to_csv(OUTPUT_DIR / "data_treatment_summary.csv", index=False,)

display(treatment_summary)

In [ ]:
clean_dbh_missing = sum(row[0] is None for row in arcpy.da.SearchCursor(TREES_CLEAN, ["DBH_CLEAN"]))
imputed_count = sum( int(row[0] or 0) for row in arcpy.da.SearchCursor(TREES_CLEAN, ["DBH_IMPUTED"]))

print(f"Final asset count:        {int(arcpy.management.GetCount(TREES_CLEAN)[0]):,}")
print(f"DBH values imputed:       {imputed_count:,}")
print(f"DBH_CLEAN missing values: {clean_dbh_missing:,}")
print(f"Output feature class:     {TREES_CLEAN}")

if clean_dbh_missing:
    raise RuntimeError("DBH_CLEAN still contains missing values.")

print("Notebook 2 completed successfully.")